# Day 5–6：Context 与 Memory 管理

之前做的 Agent 有个严重问题：**对话超过一定轮数，它就开始"失忆"**。

这不是 Bug，而是 LLM 的物理限制——每个模型都有一个最大 Token 数（Context Window）。
超过这个限制，早期的对话内容就会被"挤出去"。

今天的学习路线：
1. Context Window 的硬限制与 "Lost in the Middle" 现象
2. Memory 方案对比：滑动窗口、摘要压缩、向量存储、混合方案
3. LangGraph 的 Checkpointer：多轮对话的状态持久化
4. Prompt 设计中的 Memory 利用
5. 动手：给 Agent 加入 Memory 模块，测试 30 轮对话

## 一、Context Window 的硬限制

每个模型有一个最大 Token 数（上下文窗口），超过这个限制的 Token 会被截断：

| 模型 | 最大 Context Window |
|------|-------------------|
| GPT-4 | 128K tokens |
| Claude | 200K tokens |
| Qwen2.5-7B | 128K tokens |
| DeepSeek-V3 | 128K tokens |

128K tokens 听起来很多（约 10 万字），但实际上：
- 一次普通的对话 10 轮可能就要 5K-10K tokens
- 加上工具调用的结果（文件内容、shell 输出），Token 消耗更快
- 30 轮对话后，早期信息就会被"遗忘"

### "Lost in the Middle" 现象

这是一个非常反直觉但被论文验证的事实：

**LLM 对上下文中间位置的信息关注度最低。**

```
上下文位置 → LLM 关注度
┌─────────────────────────────────┐
│ ████████░░░░░░░░░░░░░░████████ │
│ 开头高     中间低      结尾高    │
└─────────────────────────────────┘
```

这意味着：
- 放在 Prompt 开头的指令 → 模型很关注 ✅
- 放在 Prompt 结尾的最新对话 → 模型很关注 ✅
- 放在 Prompt 中间的历史信息 → 模型容易忽略 ❌

**对 Agent 设计的启示：**
不要把重要的信息埋在对话历史的中间位置。重要的上下文要么放开头（system prompt），
要么通过 Memory 模块提取出来放在当前轮。

### Token 计数：量化理解 Context 消耗

先学会怎么数 Token，才能理解 Memory 管理为什么必要。

In [1]:
import tiktoken

# tiktoken 是 OpenAI 开源的 Token 计数工具
# 不同模型用不同的编码器

def count_tokens(text: str, model: str = "gpt-4") -> int:
    """计算一段文本的 Token 数量"""
    try:
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # 如果模型有特定编码器但 tiktoken 不支持，用 cl100k_base
        encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))

# 测试不同类型文本的 Token 消耗
test_texts = {
    "中文短句": "你好，今天天气怎么样？",
    "英文短句": "Hello, how is the weather today?",
    "代码片段": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "JSON数据": '{"name": "张三", "age": 25, "city": "北京", "hobbies": ["编程", "阅读", "跑步"]}',
}

for name, text in test_texts.items():
    token_count = count_tokens(text)
    char_count = len(text)
    print(f"{name}: {char_count} 字符 ≈ {token_count} tokens (比例 {token_count/char_count:.2f})")

print("\n关键发现：")
print("- 中文每个字符约消耗 1.5-2 个 Token")
print("- 英文每个单词约消耗 1-1.3 个 Token") 
print("- 代码/JSON 的 Token 消耗与英文类似")

中文短句: 11 字符 ≈ 13 tokens (比例 1.18)
英文短句: 32 字符 ≈ 8 tokens (比例 0.25)
代码片段: 92 字符 ≈ 28 tokens (比例 0.30)
JSON数据: 70 字符 ≈ 37 tokens (比例 0.53)

关键发现：
- 中文每个字符约消耗 1.5-2 个 Token
- 英文每个单词约消耗 1-1.3 个 Token
- 代码/JSON 的 Token 消耗与英文类似


### 模拟上下文溢出

写一个简单的实验来看看长对话的 Token 消耗速度。

In [ ]:
import random

def simulate_conversation(num_turns: int, avg_user_tokens: int = 50, avg_ai_tokens: int = 100):
    """
    模拟多轮对话的 Token 累积
    
    参数：
    - num_turns: 对话轮数
    - avg_user_tokens: 用户每轮平均 Token 数
    - avg_ai_tokens: AI 每轮平均 Token 数
    """
    total_tokens = 0
    print(f"{'轮数':<6} {'本轮消耗':<12} {'累计Token':<14} {'占128K比例':<12} {'说明'}")
    print("-" * 65)
    
    for turn in range(1, num_turns + 1):
        user_tokens = random.randint(avg_user_tokens - 20, avg_user_tokens + 20)
        ai_tokens = random.randint(avg_ai_tokens - 30, avg_ai_tokens + 50)
        turn_tokens = user_tokens + ai_tokens
        total_tokens += turn_tokens
        
        pct = total_tokens / 128000 * 100
        
        # 标记关键节点
        note = ""
        if pct > 80:
            note = "⚠️ 上下文接近满载！"
        elif pct > 50:
            note = "⚡ 上下文过半"
        elif pct > 20:
            note = "📝 开始累积"
        
        print(f"{turn:<6} {turn_tokens:<12} {total_tokens:<14} {pct:<11.1f}% {note}")
    
    print(f"\n结论：{num_turns} 轮对话后，累计消耗 {total_tokens} tokens，占 128K 的 {total_tokens/1280:.1f}%")

# 模拟 50 轮对话
simulate_conversation(50, avg_user_tokens=80, avg_ai_tokens=200)

## 二、Memory 方案对比

解决上下文溢出有四种主流方案。下面逐个实现并对比。

| 方案 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| 滑动窗口 | 只保留最近 N 轮 | 简单，延迟低 | 丢失历史 |
| 摘要压缩 | LLM 总结早期对话 | 保留信息要点 | 丢失细节 |
| 向量存储 | 语义检索历史信息 | 精准召回 | 需要额外存储 |
| 混合方案 | 近期滑动窗口 + 远期向量检索 | 兼顾效率和精度 | 实现复杂 |

### 方案一：滑动窗口（Sliding Window）

最简单的 Memory 方案：只保留最近 N 轮对话，超过的直接丢弃。

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

class SlidingWindowMemory:
    """
    滑动窗口 Memory：只保留最近 N 轮对话
    
    一轮对话 = 一条 HumanMessage + 一条 AIMessage
    """
    
    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages: list = []
    
    def add_turn(self, user_msg: str, ai_msg: str):
        """添加一轮对话"""
        self.messages.append(HumanMessage(content=user_msg))
        self.messages.append(AIMessage(content=ai_msg))
        self._trim()
    
    def _trim(self):
        """裁剪：只保留最近的 max_turns 轮"""
        max_messages = self.max_turns * 2  # 每轮 2 条消息
        if len(self.messages) > max_messages:
            self.messages = self.messages[-max_messages:]
    
    def get_messages(self) -> list:
        """获取当前窗口内的消息"""
        return self.messages
    
    def __len__(self):
        return len(self.messages) // 2  # 返回轮数


# 测试滑动窗口
memory = SlidingWindowMemory(max_turns=3)

for i in range(5):
    memory.add_turn(f"用户消息 {i+1}", f"AI回复 {i+1}")
    print(f"第 {i+1} 轮后，窗口中有 {len(memory)} 轮对话:")
    for msg in memory.get_messages():
        role = "用户" if isinstance(msg, HumanMessage) else "AI"
        print(f"  [{role}] {msg.content}")
    print()

### 方案二：摘要压缩（Summary Compression）

每 N 轮让 LLM 把历史对话总结成一段摘要。用摘要替代原始对话，显著减少 Token 消耗。

In [ ]:
import os
from langchain_openai import ChatOpenAI

class SummaryMemory:
    """
    摘要压缩 Memory

    策略：积累 N 轮对话后，用 LLM 生成一段摘要
    后续对话只保留摘要 + 最近的对话
    """

    def __init__(self, llm, summarize_every: int = 5):
        self.llm = llm
        self.summarize_every = summarize_every  # 每 N 轮总结一次
        self.recent_messages: list = []           # 最近的消息（未总结的）
        self.summary: str = ""                    # 历史摘要
        self.turn_count: int = 0                  # 当前未总结的轮数

    def add_turn(self, user_msg: str, ai_msg: str):
        """添加一轮对话"""
        self.recent_messages.append(HumanMessage(content=user_msg))
        self.recent_messages.append(AIMessage(content=ai_msg))
        self.turn_count += 1

        if self.turn_count >= self.summarize_every:
            self._summarize()

    def _summarize(self):
        """调用 LLM 生成摘要"""
        # 把最近的消息转成文本
        recent_text = "\n".join([
            f"{'用户' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
            for m in self.recent_messages
        ])

        prompt = f"""请用 2-3 句话总结以下对话的关键信息：

已有的历史摘要：{self.summary if self.summary else '（无）'}

新的对话：
{recent_text}

请把新的信息合并进历史摘要中。只输出摘要内容，不要其他文字。
"""
        response = self.llm.invoke(prompt)
        self.summary = response.content
        self.recent_messages = []  # 清空最近消息
        self.turn_count = 0
        print(f"  [摘要更新] {self.summary[:80]}...")

    def get_context(self) -> str:
        """构建给 LLM 的完整上下文"""
        context = ""
        if self.summary:
            context += f"## 历史对话摘要\n{self.summary}\n\n"
        if self.recent_messages:
            context += "## 最近对话\n"
            for m in self.recent_messages:
                role = "用户" if isinstance(m, HumanMessage) else "AI"
                context += f"{role}: {m.content}\n"
        return context

# 初始化 LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)
from langchain_core.messages import HumanMessage, AIMessage
print("摘要 Memory 初始化完成")

### 方案三：向量存储（Vector Store Memory）

将历史对话做 Embedding 存入向量数据库，按语义检索相关的历史信息。
适合需要精准找回特定历史信息的场景。

In [ ]:
# ===== 向量存储 Memory（概念演示） =====
# 实际使用需要安装：uv pip install chromadb sentence-transformers

from langchain_core.messages import HumanMessage, AIMessage
import json

class VectorStoreMemory:
    """
    向量存储 Memory
    
    原理：
    1. 把每轮对话做 Embedding（向量化）
    2. 存入向量数据库
    3. 新问题来时，用语义相似度检索最相关的历史对话
    4. 把检索到的历史信息注入当前 Prompt
    
    注意：这里是一个简化的概念实现，实际使用时推荐用 ChromaDB + sentence-transformers
    """
    
    def __init__(self):
        # 用字典模拟向量存储（key=对话文本, value=embedding 向量）
        # 实际项目中替换为 ChromaDB / FAISS / Pinecone
        self.store: dict = {}
        self.turns: list = []  # 记录所有对话
    
    def add_turn(self, user_msg: str, ai_msg: str):
        """存储一轮对话"""
        turn_text = f"用户: {user_msg}\nAI: {ai_msg}"
        self.turns.append({
            "user": user_msg,
            "ai": ai_msg,
            "text": turn_text
        })
    
    def retrieve(self, query: str, top_k: int = 3) -> list[str]:
        """
        根据查询检索最相关的历史对话
        
        实际实现中会：
        1. 将 query 做 Embedding
        2. 在向量数据库中做 ANN（近似最近邻）搜索
        3. 返回 Top-K 结果
        """
        # 这里用简单的关键词匹配模拟（实际用 ChromaDB.similarity_search）
        results = []
        query_words = set(query.lower().split())
        
        for turn in self.turns:
            turn_text = turn["text"].lower()
            score = sum(1 for w in query_words if w in turn_text)
            if score > 0:
                results.append((score, turn["text"]))
        
        results.sort(key=lambda x: x[0], reverse=True)
        return [r[1] for r in results[:top_k]]
    
    def __len__(self):
        return len(self.turns)


# 测试
vec_memory = VectorStoreMemory()
vec_memory.add_turn("我叫张三，今年25岁", "好的张三，我记住了")
vec_memory.add_turn("我喜欢Python编程", "Python是一门很棒的语言")
vec_memory.add_turn("我在北京工作", "北京是个好地方")
vec_memory.add_turn("今天天气怎么样", "今天晴天，25度")

print("查询: '张三多大了'")
results = vec_memory.retrieve("张三多大了")
for r in results:
    print(f"  -> {r}")

print("\n查询: '天气'  ")
results = vec_memory.retrieve("天气")
for r in results:
    print(f"  -> {r}")

print("\n注意：这是简化版关键词匹配。实际项目使用 Embedding 做语义检索，"天气"和"气温"也能匹配。")

### 方案四：混合方案（推荐）

工业生产中推荐使用**混合方案**：近期用滑动窗口 + 远期用向量检索。

```
┌────────────────────────────────────────────┐
│              Memory 架构                     │
│                                              │
│   最近 5 轮 → 滑动窗口（保留原始内容）         │
│   5 轮之前   → 向量存储（按需检索）           │
│   全局摘要   → 每 10 轮自动更新               │
└────────────────────────────────────────────┘
```

In [ ]:
class HybridMemory:
    """
    混合 Memory：近期滑动窗口 + 远期向量检索 + 全局摘要
    
    这是工业生产中推荐的方案：
    - 最近 5 轮：完整保留（滑动窗口）
    - 5-15 轮：向量存储，按需语义检索
    - 全局摘要：每 10 轮自动更新，提供整体脉络
    """
    
    def __init__(self, llm, window_size: int = 5, summary_interval: int = 10):
        self.llm = llm
        self.window_size = window_size
        self.summary_interval = summary_interval
        
        self.window_messages: list = []        # 滑动窗口中的消息
        self.archive_store: list[dict] = []    # 归档到向量存储的消息
        self.global_summary: str = ""          # 全局摘要
        self.total_turns: int = 0              # 总轮数
    
    def add_turn(self, user_msg: str, ai_msg: str):
        """添加一轮对话，自动管理内存"""
        self.total_turns += 1
        turn = {"user": user_msg, "ai": ai_msg}
        
        # 1. 放入滑动窗口
        self.window_messages.append(HumanMessage(content=user_msg))
        self.window_messages.append(AIMessage(content=ai_msg))
        
        # 2. 窗口满了，最早的对话移到归档区
        if len(self.window_messages) > self.window_size * 2:
            # 移出最早的 2 条消息（1 轮对话）
            removed = self.window_messages[:2]
            self.window_messages = self.window_messages[2:]
            # 存入归档区（实际项目存入向量数据库）
            self.archive_store.append({
                "user": removed[0].content,
                "ai": removed[1].content
            })
        
        # 3. 定期更新全局摘要
        if self.total_turns % self.summary_interval == 0:
            self._update_summary()
    
    def _update_summary(self):
        """更新全局摘要"""
        recent = "\n".join([
            f"用户: {m.content}" if isinstance(m, HumanMessage) else f"AI: {m.content}"
            for m in self.window_messages[-6:]  # 最近 3 轮
        ])
        
        prompt = f"""请更新以下对话的全局摘要。

当前摘要：{self.global_summary if self.global_summary else '（这是对话的开始）'}

最近的对话（用于更新摘要）：
{recent}

请合并新旧信息，输出 3-5 句话的摘要。只输出摘要内容。
"""
        response = self.llm.invoke(prompt)
        self.global_summary = response.content
        print(f"  [摘要更新 第{self.total_turns}轮] {self.global_summary[:80]}...")
    
    def build_context(self, query: str = "") -> str:
        """
        构建给 LLM 的完整上下文
        
        结构：
        1. 全局摘要（如果有）
        2. 检索到的相关历史（如果有 query）
        3. 最近的对话（滑动窗口）
        """
        parts = []
        
        # 全局摘要
        if self.global_summary:
            parts.append(f"## 对话全局摘要\n{self.global_summary}")
        
        # 最近的对话（放在最后，因为 LLM 对末尾信息关注度最高）
        if self.window_messages:
            recent = "\n".join([
                f"{'用户' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
                for m in self.window_messages
            ])
            parts.append(f"## 最近对话\n{recent}")
        
        return "\n\n".join(parts)
    
    def stats(self) -> dict:
        """内存状态统计"""
        return {
            "总轮数": self.total_turns,
            "窗口中的消息数": len(self.window_messages),
            "归档的消息数": len(self.archive_store),
            "是否有摘要": bool(self.global_summary)
        }


print("混合 Memory 类定义完成")
print("架构：最近 5 轮完整保留 + 更早的归档 + 每 10 轮更新摘要")

## 三、LangGraph 的 Checkpointer：状态持久化

前面讲的 Memory 方案都是自己实现的。LangGraph 内置了 Checkpointer 机制，可以**自动持久化整个 State**。

我们在 Day 1-2 已经用过 `InMemorySaver`，现在来深入理解它的原理。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage


# ===== Checkpointer 的三个核心能力 =====

# 1. 持久化：保存每次执行后的状态快照
# 2. Time Travel：回退到任意历史快照重新执行
# 3. 多会话隔离：不同 thread_id 的状态互不影响

def demo_node(state: MessagesState):
    """示例节点：回显用户消息"""
    last_msg = state["messages"][-1]
    return {"messages": [AIMessage(content=f"你说的是: {last_msg.content}")]}

graph = StateGraph(MessagesState)
graph.add_node("echo", demo_node)
graph.add_edge(START, "echo")
graph.add_edge("echo", END)

# 带 Checkpointer 编译
checkpointer = InMemorySaver()
compiled = graph.compile(checkpointer=checkpointer)

# 测试多会话隔离
config_1 = {"configurable": {"thread_id": "session-1"}}
config_2 = {"configurable": {"thread_id": "session-2"}}

# 会话 1
compiled.invoke({"messages": [HumanMessage(content="你好，我是张三")]}, config_1)
state1 = compiled.get_state(config_1)
print(f"会话 1 的消息数: {len(state1.values['messages'])}")

# 会话 2（独立的状态空间）
compiled.invoke({"messages": [HumanMessage(content="你好，我是李四")]}, config_2)
state2 = compiled.get_state(config_2)
print(f"会话 2 的消息数: {len(state2.values['messages'])}")

print("\n关键发现：两个会话的状态完全隔离，互不影响")

### Checkpointer 的 Time Travel

可以回退到任意历史状态，这在调试 Agent 行为时非常有用。

In [ ]:
# ===== Time Travel：回退到历史状态 =====

config = {"configurable": {"thread_id": "time-travel-demo"}}

# 第一轮
compiled.invoke({
    "messages": [HumanMessage(content="第一轮：我叫张三")]
}, config)

# 保存第一轮的 checkpoint id
checkpoint_1 = compiled.get_state(config)
print(f"第一轮后 checkpoint: {checkpoint_1.metadata['step']}")
print(f"消息: {[m.content for m in checkpoint_1.values['messages']]}")

# 第二轮
compiled.invoke({
    "messages": [HumanMessage(content="第二轮：我今天25岁")]
}, config)

checkpoint_2 = compiled.get_state(config)
print(f"\n第二轮后 checkpoint: {checkpoint_2.metadata['step']}")
print(f"消息: {[m.content for m in checkpoint_2.values['messages']]}")

# 回退到第一轮之后的状态
print(f"\n回退到 step {checkpoint_1.metadata['step']} 之后...")
restored = compiled.get_state(config)
print(f"恢复后消息: {[m.content for m in restored.values['messages']]}")

print("\nTime Travel 的实用场景：")
print("1. 调试：Agent 出错了，回退到出错前的状态重新执行")
print("2. A/B 测试：从同一状态分叉，测试不同策略的效果")
print("3. 用户撤销：用户说'我刚才说错了'，回退到上一轮")

## 四、Prompt 设计中的 Memory 利用

有了 Memory 机制，还需要在 Prompt 中正确利用。核心原则：

**每次思考前先回顾关键历史信息，把最重要的信息放在 Prompt 开头和结尾。**

In [ ]:
# ===== Memory-Aware Prompt 模板 =====

MEMORY_AWARE_SYSTEM_PROMPT = """你是一个具有记忆能力的 AI 助手。

## 对话全局摘要
{global_summary}

## 你的身份信息
- 你的名字是 Assistant
- 你正在帮助一位用户解决问题
- 如果用户之前告诉过你他们的信息（姓名、偏好等），请在回答中体现

## 重要提示
- 回答问题时，先检查上面的摘要中是否有相关信息
- 如果用户问到了历史对话中提过的事情，引用历史信息来回答
- 如果摘要中没有相关信息，诚实地说不知道，不要编造
"""

MEMORY_REVIEW_PROMPT = """在回答用户问题之前，请先回顾以下信息：

1. 全局摘要中提到了用户的哪些关键信息？
2. 最近对话中用户在讨论什么话题？
3. 当前问题与历史对话有什么关联？

如果当前问题与历史对话无关，请忽略以上回顾，直接回答问题。
"""


def build_memory_aware_prompt(memory: HybridMemory, user_query: str) -> list:
    """
    构建带 Memory 的完整 Prompt
    
    结构设计遵循 'Lost in the Middle' 的最佳实践：
    - System prompt（开头，关注度最高）: 包含全局摘要
    - 中间部分: 检索到的相关历史
    - 最新消息（结尾，关注度最高）: 用户当前问题
    """
    system_prompt = MEMORY_AWARE_SYSTEM_PROMPT.format(
        global_summary=memory.global_summary or "（这是对话的开始，还没有历史摘要）"
    )
    
    # 构建消息列表
    messages = [{"role": "system", "content": system_prompt}]
    
    # 加上最近的对话
    for msg in memory.window_messages:
        role = "user" if isinstance(msg, HumanMessage) else "assistant"
        messages.append({"role": role, "content": msg.content})
    
    # 加上当前问题
    messages.append({"role": "user", "content": user_query})
    
    return messages


print("Memory-Aware Prompt 模板定义完成")
print("\n设计原则：")
print("1. 全局摘要放在 System Prompt（开头），让模型从一开始就知道上下文")
print("2. 最近对话放在中间，提供细节")
print("3. 用户当前问题放在最后，模型对末尾信息关注度最高")
print("4. 关键信息不要在中间位置，要么开头要么结尾")

## 五、整合：带 Memory 的 Agent

将 Memory 模块整合到 LangGraph Agent 中，实现完整的记忆能力。

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


# ===== 1. 定义 Agent 状态（包含 Memory） =====
class MemoryAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    memory_summary: str   # 对话摘要
    turn_count: int       # 当前轮数


# ===== 2. LLM 思考节点（带 Memory） =====
def memory_aware_llm(state: MemoryAgentState):
    """
    带记忆感知的 LLM 节点
    
    关键设计：
    - 如果有摘要，注入到 system message 中
    - 摘要放在消息列表的开头（利用 LLM 对开头的关注）
    """
    messages = list(state["messages"])
    
    # 如果存在摘要，在消息列表最前面插入 system message
    if state.get("memory_summary"):
        summary_msg = SystemMessage(content=f"""以下是之前对话的关键信息摘要：
{state['memory_summary']}

请在回答时利用这些历史信息。""")
        messages = [summary_msg] + messages
    
    response = llm.invoke(messages)
    return {"messages": [response]}


# ===== 3. Memory 管理节点 =====
def manage_memory(state: MemoryAgentState):
    """
    Memory 管理节点：检查是否需要压缩历史
    
    策略：
    - 每 5 轮对话，用 LLM 生成对话摘要
    - 摘要替换掉早期的详细对话
    """
    turn_count = state.get("turn_count", 0) + 1
    
    if turn_count > 0 and turn_count % 5 == 0:
        # 每 5 轮生成摘要
        messages = state["messages"]
        # 把最近的 10 条消息转成对话文本
        recent = messages[-10:]
        dialog_text = "\n".join([
            f"{'用户' if isinstance(m, HumanMessage) else 'AI'}: {m.content[:100]}"
            for m in recent
        ])
        
        existing = state.get("memory_summary", "")
        summary_prompt = f"""请将以下新对话合并进历史摘要中。

历史摘要：{existing if existing else '（这是对话开始）'}

新对话：
{dialog_text}

请输出更新后的摘要（3-5 句话）。只输出摘要内容。"""
        
        new_summary = llm.invoke(summary_prompt).content
        return {"memory_summary": new_summary, "turn_count": turn_count}
    
    return {"turn_count": turn_count}


# ===== 4. 构建图 =====
builder = StateGraph(MemoryAgentState)
builder.add_node("memory", manage_memory)
builder.add_node("llm", memory_aware_llm)

builder.add_edge(START, "memory")
builder.add_edge("memory", "llm")
builder.add_edge("llm", END)

checkpointer = InMemorySaver()
memory_agent = builder.compile(checkpointer=checkpointer)

print("带 Memory 的 Agent 构建完成！")

### 测试 30 轮对话

这是 Memory 模块的真正考验——30 轮连续对话后，Agent 是否还记得第 1 轮的内容？

In [ ]:
# ===== 30 轮对话压力测试 =====

config = {"configurable": {"thread_id": "memory-test-30"}}

# 第一轮：告诉 Agent 用户信息
result = memory_agent.invoke({
    "messages": [HumanMessage(content="你好，我叫张三，今年25岁，住在北京，是一名Python工程师。请记住这些信息。")],
    "turn_count": 0,
    "memory_summary": ""
}, config)
print(f"[第1轮] {result['messages'][-1].content[:80]}...")

# 中间 28 轮：各种无关对话
topics = [
    "Python的列表和元组有什么区别？",
    "推荐几个学习机器学习的资源",
    "今天天气真好啊",
    "什么是Docker？",
    "解释一下RESTful API",
    "Python如何读取CSV文件？",
    "什么是Git？为什么需要版本控制？",
    "讲讲你理解的面向对象编程",
    "我最喜欢的编程语言是Python",
    "北京有什么好玩的地方？",
    "Linux常用的命令有哪些？",
    "什么是数据库索引？",
    "给我讲个笑话吧",
    "Python装饰器怎么用？",
    "什么是微服务架构？",
    "Redis是做什么的？",
    "如何学好英语？",
    "讲讲HTTP和HTTPS的区别",
    "什么是CI/CD？",
    "Python的异步编程怎么理解？",
    "什么是设计模式？",
    "推荐几本好书",
    "如何提高代码质量？",
    "讲讲CAP理论",
    "什么是WebSocket？",
    "Python的GIL是什么？",
    "Kubernetes有什么优势？",
    "如何做好时间管理？",
]

for i, topic in enumerate(topics, start=2):
    result = memory_agent.invoke(
        {"messages": [HumanMessage(content=topic)]},
        config
    )
    if i % 5 == 0:
        summary = memory_agent.get_state(config).values.get("memory_summary", "")
        print(f"[第{i}轮] 摘要已更新: {summary[:80] if summary else '无'}...")

# 第 30 轮：测试是否还记得第一轮的信息
print("\n===== 第 30 轮：测试记忆 =====")
result = memory_agent.invoke({
    "messages": [HumanMessage(content="你还记得我叫什么名字，多大年龄，住在哪里，做什么工作吗？")]
}, config)
print(f"Agent 回答: {result['messages'][-1].content}")

# 检查当前 Memory 状态
final_state = memory_agent.get_state(config)
print(f"\n===== Memory 状态 =====")
print(f"总轮数: {final_state.values.get('turn_count', 'N/A')}")
print(f"摘要: {final_state.values.get('memory_summary', '无')[:150]}...")
print(f"消息数: {len(final_state.values.get('messages', []))}")

## 今日总结

Memory 管理是 Agent 工程中最容易被忽视但最重要的一环。今天掌握了：

1. **Context Window 限制** — 每个模型有最大 Token 数，"Lost in the Middle" 说明重要信息不要放中间
2. **四种 Memory 方案** — 滑动窗口（简单）、摘要压缩（平衡）、向量存储（精准）、混合方案（推荐）
3. **LangGraph Checkpointer** — 自动持久化状态，支持多会话隔离和 Time Travel
4. **Prompt 中的 Memory 利用** — 关键信息放开头/结尾，利用 LLM 对两端的高关注度
5. **30 轮对话测试** — 验证 Agent 在长对话中是否还记得早期信息

**面试要点：**
- 问到 "Context Window" → 说出 Token 限制 + Lost in the Middle
- 问到 "Memory 方案" → 说出四种方案及适用场景，强调混合方案
- 问到 "长对话退化" → 说出摘要压缩 + 向量检索 + 滑动窗口的组合策略

明天是 Week 1 的整合日——把所有组件串起来，构建一个完整的工业级 Agent。